# Landscape Interactive Explorer

No simulation, no integration, no vector field -- just a direct query tool.
Move three sliders (pseudotime, UMAP1, UMAP2) to place a point anywhere in
the landscape; the plot updates live, and the decoded gene expression at
that exact point is recomputed and shown immediately.

**Pipeline per query:** `(t, u1, u2)` &rarr; `f_phi` (a small trained MLP)
&rarr; scVI latent vector `z` &rarr; frozen scVI decoder &rarr; gene
expression. One forward pass, no iteration -- nothing here can drift or
diverge the way the trajectory simulator could.

**Prerequisite:** this notebook loads `artifacts/adata_filtered.h5ad` and
`artifacts/scvi_model/` from an earlier run (scVI training + pseudotime
filtering). If those don't exist yet, run those two steps from the
pipeline notebook first.


## 0. Config

In [ ]:
import os

ARTIFACT_DIR = "artifacts"
SCVI_MODEL_DIR = os.path.join(ARTIFACT_DIR, "scvi_model")
FILTERED_ADATA_PATH = os.path.join(ARTIFACT_DIR, "adata_filtered.h5ad")

PSEUDOTIME_KEY = "Pseudotime"
BATCH_KEY = "replicate_id"

FPHI_HIDDEN_DIM = 128
FPHI_EPOCHS = 300
FPHI_BATCH_SIZE = 512
FPHI_LR = 1e-3
VAL_FRACTION = 0.15
RANDOM_SEED = 0

# A few marker genes to highlight if present in this dataset -- swap these
# for whatever's biologically relevant to you. If none are present, the
# explorer falls back to showing the top-expressed genes at each point.
GENES_OF_INTEREST = ["Sox2", "Olig2", "Nkx6-1", "Pax6", "Nkx2-2", "Foxa2"]


## 1. Load the filtered AnnData and the scVI model

Both were saved by the earlier pipeline notebook. `SCVI.load(..., adata=...)`
needs *some* AnnData with the same registered structure -- we pass the same
filtered one we just loaded, which is fine since it's the same population
the model was trained on.


In [ ]:
import scanpy as sc
import scvi
import numpy as np
import torch

assert os.path.exists(FILTERED_ADATA_PATH), \
    f"{FILTERED_ADATA_PATH} not found -- run the scVI training + filtering steps first."
assert os.path.exists(SCVI_MODEL_DIR), \
    f"{SCVI_MODEL_DIR} not found -- run the scVI training step first."

adata_filtered = sc.read_h5ad(FILTERED_ADATA_PATH)
print(adata_filtered)

loaded_scvi = scvi.model.SCVI.load(SCVI_MODEL_DIR)
loaded_scvi.module.eval()
print("scVI model loaded.")


## 2. Train the space field — f_phi(t, u1, u2) → scVI latent vector

Exact (cell, latent) pairs from real data, no derived/noisy targets
involved -- plain MSE regression is the right tool here, same reasoning as
the very first version of this pipeline.


In [ ]:
import torch.nn as nn
from sklearn.model_selection import train_test_split

class SpaceField(nn.Module):
    def __init__(self, latent_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
    def forward(self, t, u1, u2):
        x = torch.stack([t, u1, u2], dim=-1)
        return self.net(x)


In [ ]:
t_all = adata_filtered.obs[PSEUDOTIME_KEY].values
u_all = adata_filtered.obsm["X_umap"]
z_all = adata_filtered.obsm["X_scVI"]
latent_dim = z_all.shape[1]

t_mean_f, t_std_f = t_all.mean(), t_all.std()
u_mean_f, u_std_f = u_all.mean(axis=0), u_all.std(axis=0)

t_norm_f = (t_all - t_mean_f) / t_std_f
u_norm_f = (u_all - u_mean_f) / u_std_f

X_f = np.column_stack([t_norm_f, u_norm_f])
Y_f = z_all

X_train_f, X_val_f, Y_train_f, Y_val_f, t_train_raw_f, t_val_raw_f = train_test_split(
    X_f, Y_f, t_all, test_size=VAL_FRACTION, random_state=RANDOM_SEED
)
X_train_ft = torch.tensor(X_train_f, dtype=torch.float32)
Y_train_ft = torch.tensor(Y_train_f, dtype=torch.float32)
X_val_ft   = torch.tensor(X_val_f,   dtype=torch.float32)
Y_val_ft   = torch.tensor(Y_val_f,   dtype=torch.float32)


In [ ]:
f_phi = SpaceField(latent_dim=latent_dim, hidden_dim=FPHI_HIDDEN_DIM)
optimizer_f = torch.optim.Adam(f_phi.parameters(), lr=FPHI_LR)
loss_fn_f = nn.MSELoss()

best_val_f = float("inf")
for epoch in range(FPHI_EPOCHS):
    perm = torch.randperm(X_train_ft.shape[0])
    total_loss = 0
    for i in range(0, len(perm), FPHI_BATCH_SIZE):
        idx = perm[i:i+FPHI_BATCH_SIZE]
        xb, yb = X_train_ft[idx], Y_train_ft[idx]
        pred = f_phi(xb[:,0], xb[:,1], xb[:,2])
        loss = loss_fn_f(pred, yb)
        optimizer_f.zero_grad(); loss.backward(); optimizer_f.step()
        total_loss += loss.item() * len(idx)

    if epoch % 30 == 0 or epoch == FPHI_EPOCHS - 1:
        f_phi.eval()
        with torch.no_grad():
            val_pred = f_phi(X_val_ft[:,0], X_val_ft[:,1], X_val_ft[:,2])
            val_loss = loss_fn_f(val_pred, Y_val_ft).item()
        f_phi.train()
        if val_loss < best_val_f:
            best_val_f = val_loss
            torch.save(f_phi.state_dict(), os.path.join(ARTIFACT_DIR, "f_phi_best.pt"))
        print(f"epoch {epoch:3d} | train {total_loss/len(perm):.4f} | val {val_loss:.4f}")

np.savez(os.path.join(ARTIFACT_DIR, "norm_stats_fphi.npz"),
         t_mean=t_mean_f, t_std=t_std_f, u_mean=u_mean_f, u_std=u_std_f, latent_dim=latent_dim)
f_phi.load_state_dict(torch.load(os.path.join(ARTIFACT_DIR, "f_phi_best.pt")))
f_phi.eval()


In [ ]:
# Residual-by-region check -- same diagnostic as every previous version,
# just to know up front if any part of the landscape is less trustworthy.
with torch.no_grad():
    val_pred_f = f_phi(X_val_ft[:,0], X_val_ft[:,1], X_val_ft[:,2]).numpy()
per_sample_err_f = np.linalg.norm(val_pred_f - Y_val_f, axis=1)

for lo, hi in zip([0,20,40,60,80], [20,40,60,80,95]):
    mask = (t_val_raw_f >= lo) & (t_val_raw_f < hi)
    if mask.sum() > 0:
        print(f"t in [{lo},{hi}): n={mask.sum():4d}  mean residual={per_sample_err_f[mask].mean():.4f}")


## 3. Decode wiring — z → gene expression

Same scVI `generative()` call as every previous version, validated against
real cells before trusting it on `f_phi`'s output.


In [ ]:
adata_ref = loaded_scvi.adata
ref_batch_name = adata_ref.obs[BATCH_KEY].value_counts().idxmax()
ref_batch_code = list(adata_ref.obs[BATCH_KEY].astype("category").cat.categories).index(ref_batch_name)
median_lib_size = np.median(adata_ref.obs["lib_size"].values) if "lib_size" in adata_ref.obs.columns \
                  else np.median(np.asarray(adata_ref.X.sum(axis=1)).flatten())


def decode_latent_to_expression(z_path, batch_code=ref_batch_code, library_size=None):
    n = z_path.shape[0]
    lib = library_size if library_size is not None else median_lib_size
    z_t = torch.tensor(z_path, dtype=torch.float32)
    library_log = torch.full((n, 1), np.log(lib), dtype=torch.float32)
    batch_idx = torch.full((n, 1), batch_code, dtype=torch.long)
    with torch.no_grad():
        gen_out = loaded_scvi.module.generative(z=z_t, library=library_log, batch_index=batch_idx)
    px = gen_out["px"]
    return px.mu.detach().cpu().numpy() if hasattr(px, "mu") else px.mean.detach().cpu().numpy()


In [ ]:
# Validate on 5 real cells before trusting f_phi's output
real_idx = np.arange(5)
z_real = adata_ref.obsm["X_scVI"][real_idx]
true_batch_codes = adata_ref.obs[BATCH_KEY].astype("category").cat.codes.values[real_idx]
true_lib = (adata_ref.obs["lib_size"].values[real_idx] if "lib_size" in adata_ref.obs.columns
            else np.asarray(adata_ref.X.sum(axis=1)).flatten()[real_idx])

decoded = np.array([
    decode_latent_to_expression(z_real[i:i+1], batch_code=true_batch_codes[i], library_size=true_lib[i])[0]
    for i in range(5)
])
actual = np.asarray(adata_ref.X[real_idx].todense()) if hasattr(adata_ref.X, "todense") else adata_ref.X[real_idx]
for i in range(5):
    corr = np.corrcoef(decoded[i], actual[i])[0, 1]
    print(f"cell {i}: correlation decoded vs actual = {corr:.3f}")


## 4. The interactive explorer

Three sliders control the query point. Each move triggers:
`(t, u1, u2)` &rarr; `f_phi` &rarr; `z` &rarr; decoder &rarr; expression.

Also shown: **distance to nearest real cells in latent space** — the same
confidence check from earlier debugging, repurposed here as a live signal.
Small distance = this point's decoded state resembles real biology.
Large distance = you've moved somewhere `f_phi` is extrapolating and the
expression shown is less trustworthy.

Requires `ipywidgets` and `plotly` (`pip install ipywidgets plotly
--break-system-packages` if not already installed).


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
from scipy.spatial import cKDTree

f_stats = np.load(os.path.join(ARTIFACT_DIR, "norm_stats_fphi.npz"))
tree_latent = cKDTree(adata_filtered.obsm["X_scVI"])

sample_idx = np.random.default_rng(0).choice(len(adata_filtered), min(300, len(adata_filtered)), replace=False)
typical_spacing = np.median([np.median(tree_latent.query(adata_filtered.obsm["X_scVI"][i], k=11)[0][1:])
                              for i in sample_idx])

gene_names = adata_filtered.var_names.values
genes_present = [g for g in GENES_OF_INTEREST if g in gene_names]
print("Tracking genes:", genes_present if genes_present else "(none found -- falling back to top-expressed)")


def query_point(t, u1, u2):
    t_n  = (t  - f_stats["t_mean"]) / f_stats["t_std"]
    u1_n = (u1 - f_stats["u_mean"][0]) / f_stats["u_std"][0]
    u2_n = (u2 - f_stats["u_mean"][1]) / f_stats["u_std"][1]
    x = torch.tensor([[t_n, u1_n, u2_n]], dtype=torch.float32)
    with torch.no_grad():
        z = f_phi(x[:,0], x[:,1], x[:,2]).numpy()[0]
    expr = decode_latent_to_expression(z[None, :])[0]
    dist, _ = tree_latent.query(z, k=10)
    return expr, np.median(dist)


In [ ]:
# Background scatter (downsampled for plot performance)
bg_idx = np.random.default_rng(0).choice(len(adata_filtered), min(4000, len(adata_filtered)), replace=False)
bg_t = adata_filtered.obs[PSEUDOTIME_KEY].values[bg_idx]
bg_u = adata_filtered.obsm["X_umap"][bg_idx]

t_min, t_max = adata_filtered.obs[PSEUDOTIME_KEY].min(), adata_filtered.obs[PSEUDOTIME_KEY].max()
u1_min, u1_max = adata_filtered.obsm["X_umap"][:,0].min(), adata_filtered.obsm["X_umap"][:,0].max()
u2_min, u2_max = adata_filtered.obsm["X_umap"][:,1].min(), adata_filtered.obsm["X_umap"][:,1].max()

t0_init  = float(np.median(bg_t))
u10_init = float(np.median(bg_u[:,0]))
u20_init = float(np.median(bg_u[:,1]))

if genes_present:
    show_idx = [list(gene_names).index(g) for g in genes_present]


In [ ]:
# NOTE: plotly's go.FigureWidget (>=6.0) depends on the "anywidget" JS
# module, which has a known, currently-unresolved bug in VS Code's Jupyter
# renderer ("No version of module anywidget is registered"). To avoid it
# entirely, we don't use FigureWidget's live-patching at all -- instead we
# rebuild a plain go.Figure on every slider move and redraw it inside an
# ipywidgets.Output(). Plain figures render through a different path that
# doesn't touch anywidget, so this sidesteps the bug rather than fighting
# version pinning. (If you'd rather keep the smoother live-patch behavior,
# the alternative fix is `pip install "plotly<6"` instead of this redraw
# approach -- either works, this one is just more robust to environment
# quirks.)
from plotly.subplots import make_subplots
from IPython.display import clear_output

plot_output = widgets.Output()


def make_combined_figure(t, u1, u2):
    expr, conf_dist = query_point(t, u1, u2)
    if genes_present:
        vals, names = expr[show_idx], genes_present
    else:
        order = np.argsort(expr)[-15:]
        vals, names = expr[order], gene_names[order]

    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "xy"}]],
                         column_widths=[0.55, 0.45],
                         subplot_titles=("Query point in the landscape", "Decoded expression here"))

    fig.add_trace(go.Scatter3d(x=bg_t, y=bg_u[:,0], z=bg_u[:,1], mode="markers",
                                marker=dict(size=2, color=bg_t, colorscale="Viridis", opacity=0.35),
                                hoverinfo="skip", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[t], y=[u1], z=[u2], mode="markers",
                                marker=dict(size=8, color="red", line=dict(color="white", width=1)),
                                showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(x=vals, y=names, orientation="h", marker_color="#f0a05a", showlegend=False),
                  row=1, col=2)

    fig.update_layout(height=460, margin=dict(l=10, r=10, t=40, b=10),
                       scene=dict(xaxis_title="pseudotime", yaxis_title="UMAP1", zaxis_title="UMAP2"))
    fig.update_xaxes(title_text="decoded expression", row=1, col=2)

    flag = "good -- near real data" if conf_dist < typical_spacing * 2 else "caution -- far from real data, extrapolating"
    info_html = (f"<b>Distance to nearest real cells (latent space):</b> {conf_dist:.3f} "
                 f"(typical real-cell spacing: {typical_spacing:.3f}) &nbsp; [{flag}]")
    return fig, info_html


In [ ]:
t_slider  = widgets.FloatSlider(value=t0_init,  min=float(t_min),  max=float(t_max),  step=(t_max-t_min)/300,   description="pseudotime", continuous_update=True, readout_format=".2f")
u1_slider = widgets.FloatSlider(value=u10_init, min=float(u1_min), max=float(u1_max), step=(u1_max-u1_min)/300, description="UMAP1",     continuous_update=True, readout_format=".2f")
u2_slider = widgets.FloatSlider(value=u20_init, min=float(u2_min), max=float(u2_max), step=(u2_max-u2_min)/300, description="UMAP2",     continuous_update=True, readout_format=".2f")


def update(change=None):
    fig, info_html = make_combined_figure(t_slider.value, u1_slider.value, u2_slider.value)
    with plot_output:
        clear_output(wait=True)
        display(widgets.HTML(info_html))
        fig.show()

t_slider.observe(update, names="value")
u1_slider.observe(update, names="value")
u2_slider.observe(update, names="value")

display(widgets.VBox([widgets.HBox([t_slider, u1_slider, u2_slider]), plot_output]))
update()


## 6. Manifold-constrained greedy walk

A simpler alternative way to trace a path through the landscape, evaluated
entirely through `f_phi`'s smooth output rather than the MDN/OT machinery
from earlier notebooks.

**The algorithm, per step:** sample candidate `(u1, u2)` points in a radius
around the current position, at the next pseudotime step. Decode each
candidate through `f_phi` to get its latent vector. **Reject any candidate
whose decoded latent vector is too far from real cells** (the same
nearest-real-cell-distance check from the interactive explorer above) --
this is the anchor that was missing from the original proposal. Among the
remaining *valid* candidates, move to whichever one is closest **in latent
space** to the current point (smoothest possible step) -- this replaces
"most correlated in raw expression," which is a noisier signal since most
genes are near-zero in any given cell and can inflate correlation
artificially.

Built with a toggle so you can directly compare constrained vs.
unconstrained -- that comparison is the actual evidence for whether the
anchor matters, not just an assertion.


In [ ]:
def greedy_manifold_walk(t0, u1_0, u2_0, dt=0.5, search_radius=2.0, n_candidates=60,
                          manifold_threshold_mult=2.0, use_manifold_constraint=True,
                          t_end=None, seed=0):
    rng = np.random.default_rng(seed)
    t_end_local = t_max if t_end is None else t_end
    threshold = typical_spacing * manifold_threshold_mult

    t, u1, u2 = t0, u1_0, u2_0
    path_t, path_u1, path_u2, path_conf = [t], [u1], [u2], [0.0]

    def get_z(t_arr, u1_arr, u2_arr):
        t_n  = (t_arr  - f_stats["t_mean"]) / f_stats["t_std"]
        u1_n = (u1_arr - f_stats["u_mean"][0]) / f_stats["u_std"][0]
        u2_n = (u2_arr - f_stats["u_mean"][1]) / f_stats["u_std"][1]
        x = torch.tensor(np.column_stack([t_n, u1_n, u2_n]), dtype=torch.float32)
        with torch.no_grad():
            return f_phi(x[:,0], x[:,1], x[:,2]).numpy()

    z_current = get_z(np.array([t]), np.array([u1]), np.array([u2]))[0]

    while t < t_end_local:
        t_next = min(t + dt, t_end_local)

        angles = rng.uniform(0, 2*np.pi, n_candidates)
        radii  = rng.uniform(0, search_radius, n_candidates)
        cand_u1 = u1 + radii * np.cos(angles)
        cand_u2 = u2 + radii * np.sin(angles)
        cand_t  = np.full(n_candidates, t_next)

        z_cands = get_z(cand_t, cand_u1, cand_u2)
        dist_to_real = tree_latent.query(z_cands, k=10)[0].mean(axis=1)

        if use_manifold_constraint:
            valid = dist_to_real < threshold
            if not valid.any():
                print(f"t={t:.1f}: no on-manifold candidates in radius -- stopping early.")
                break
        else:
            valid = np.ones(n_candidates, dtype=bool)   # no anchor -- for comparison only

        latent_dist = np.linalg.norm(z_cands[valid] - z_current, axis=1)
        best_local = np.argmin(latent_dist)
        chosen = np.where(valid)[0][best_local]

        u1, u2, t = cand_u1[chosen], cand_u2[chosen], t_next
        z_current = z_cands[chosen]
        path_t.append(t); path_u1.append(u1); path_u2.append(u2)
        path_conf.append(dist_to_real[chosen])

    return np.array(path_t), np.array(path_u1), np.array(path_u2), np.array(path_conf)


In [ ]:
# Run both versions from the same real, low-pseudotime starting cell
early_mask_w = adata_filtered.obs[PSEUDOTIME_KEY] < 5
early_idx_w = np.where(early_mask_w.values)[0][0]
wt0  = float(adata_filtered.obs[PSEUDOTIME_KEY].values[early_idx_w])
wu1_0 = float(adata_filtered.obsm["X_umap"][early_idx_w, 0])
wu2_0 = float(adata_filtered.obsm["X_umap"][early_idx_w, 1])

pt_c, pu1_c, pu2_c, conf_c = greedy_manifold_walk(wt0, wu1_0, wu2_0, use_manifold_constraint=True)
pt_u, pu1_u, pu2_u, conf_u = greedy_manifold_walk(wt0, wu1_0, wu2_0, use_manifold_constraint=False)

print("Constrained walk steps:", len(pt_c), "| final confidence dist:", conf_c[-1])
print("Unconstrained walk steps:", len(pt_u), "| final confidence dist:", conf_u[-1])
print("Typical real-cell spacing for reference:", typical_spacing)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

fig = plt.figure(figsize=(13, 5))

ax = fig.add_subplot(1, 2, 1, projection="3d")
bg_t_w = adata_filtered.obs[PSEUDOTIME_KEY].values
bg_u_w = adata_filtered.obsm["X_umap"]
ax.scatter(bg_t_w, bg_u_w[:,0], bg_u_w[:,1], s=2, alpha=0.10, c=bg_t_w, cmap="viridis")
ax.plot(pt_c, pu1_c, pu2_c, color="lime", linewidth=2, label="constrained")
ax.plot(pt_u, pu1_u, pu2_u, color="red", linewidth=2, label="unconstrained")
ax.set_xlabel("pseudotime"); ax.set_ylabel("UMAP1"); ax.set_zlabel("UMAP2")
ax.legend(); ax.set_title("Constrained (green) vs unconstrained (red) walk")

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(pt_c, conf_c, color="lime", label="constrained")
ax2.plot(pt_u, conf_u, color="red", label="unconstrained")
ax2.axhline(typical_spacing, color="gray", linestyle="--", label="typical real-cell spacing")
ax2.set_xlabel("pseudotime"); ax2.set_ylabel("distance to nearest real cells")
ax2.legend(); ax2.set_title("Does the unconstrained walk drift off-manifold?")

plt.tight_layout()
plt.show()


### What to look for

If the red (unconstrained) line in the right-hand plot climbs steadily
above the gray "typical spacing" reference while green stays near it, that
directly confirms the concern from before: nothing stops a pure
latent-distance-minimizing walk from drifting into territory `f_phi` is
extrapolating, since smoothness alone was never evidence of correctness.
If both stay low and close together, the anchor wasn't doing much work for
*this particular* starting point and trajectory -- worth trying a few
different starting points before concluding either way.


### Notes

- Sliders are bounded to the **observed data range** by construction, so
  every reachable point is at least coordinate-wise inside the landscape.
  That alone doesn't guarantee the point sits *on* the real manifold (a
  point can be inside the bounding box but in empty space the data curves
  around) — which is exactly what the confidence readout is for. If it
  flags orange, treat the decoded expression as a rough extrapolation, not
  a real cell state.
- Want a literal click-to-place point instead of sliders? That's possible
  with Plotly's `on_click` callback on the 3D scene, but click resolution
  in a rotated 3D view is unreliable for picking exact coordinates —
  sliders give you precise, repeatable control, which matters more for
  actually reading off expression values than free-form clicking would.
